**Executing this notebook to check zip file is good and extracting to right folder**

In [1]:
%run ../utility/common_functions.ipynb

In [ ]:

# Read inbound folder from configuration
# Convert configured stage reference into Snowflake stage path
input_stage_path = input_folder.replace("@AZURE_SNOW_STAGE",f"@{stage_name}")

# Discover ZIP files dynamically
zip_files = session.sql(f"LIST {input_stage_path}").collect()

# Extract only the file names, not the complete Azure URLs
zip_files = [
    row["name"].rsplit("/", 1)[-1]
    for row in zip_files
    if row["name"].lower().endswith(".zip")
]
print("ZIP files discovered:", len(zip_files))
for file_name in zip_files:
    print("ZIP filename:", file_name)

In [2]:
folder_name = datetime.now().strftime("%Y%m%d")
run_start_time = datetime.now()
run_id = (f"{datetime.now().strftime('%Y%m%d_%H%M%S')}")
pipeline_name = "Customer Billing Pipeline"
task_name = "Stage 1 Unzip Source File"

In [3]:
final_input_folder = (input_stage_path.rstrip("/") + "/" + file_name)
print("Final Snowflake ZIP path:", final_input_folder)

In [ ]:
# Stage 1: ZIP extraction with ETL logging
validation_message = "ZIP processing has not started"

# Expected file count from configuration
expected_files = int(expected_files)

try:
    # 1. Log execution start
    process_log("Start",file_name=file_name,layer="BRONZE")

    # 2. Validate and extract ZIP
    validation_message = unzip_files(final_input_folder,unzip_folder,expected_files)
    if "Error :" in validation_message:
        raise Exception(validation_message)
    print("Validation message after execution:", validation_message)

    # 3. Log successful execution
    process_log("Success",file_name=file_name,layer="BRONZE")

except Exception as e:
    # 4. Log failure
    process_log("Failed",file_name=file_name,layer="BRONZE",error_message=str(e))
    raise

In [ ]:
# List Bronze YAML configuration files from Snowflake stage
session.sql("USE DATABASE CUSTOMER_BILLING_DB").collect()
final_unzip_path = unzip_folder.rstrip("/")
all_files = session.sql(f"LIST {bronze_yaml_path}").collect()
for actual_file_path in all_files:
    yaml_file_name = actual_file_path["name"].rsplit("/", 1)[-1]
    if not yaml_file_name.lower().endswith((".yaml", ".yml")):
        continue

    run_id = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    absolute_path = bronze_yaml_path.rstrip("/") + "/" + yaml_file_name
    table_name = yaml_file_name.rsplit(".", 1)[0]

    try:
        # Read YAML configuration
        yaml_content = open_file_read_data(absolute_path)
        reading_yaml_data = yaml.safe_load(yaml_content)

        ytable_name = reading_yaml_data["table"]["name"]
        ytable_schema_name = reading_yaml_data["table"]["schema_name"]
        final_table_name = ytable_schema_name + "." + ytable_name
        ytable_source_file = reading_yaml_data["table"]["source_file"]
        ytable_columns_names = reading_yaml_data["table"]["columns"]

        # File name pattern
        file_name_prefix = ytable_source_file.split("*")[0]
        file_name_pattern = rf"^{file_name_prefix}_(\d{{8}})_(\d{{6}})\.csv$"

        # Recursive search
        final_file_name, final_file_path = read_recursive_file_data(final_unzip_path)
        print("final_file_name =>", final_file_name)
        print("final_file_path =>", final_file_path)

        if not final_file_name or not final_file_path:
            raise FileNotFoundError(f"No matching CSV file found for table: {table_name}")

        #Get the timestamp of extract folder
        extract_folder = final_file_path.rsplit("/", 1)[0]
        print(f"extract_folder: {extract_folder}")
        #summary file path
        summary_file_path = extract_folder + "/summary_file.txt"
        print(f"summary_file_path: {summary_file_path}")

        # START LOG
        process_log("Start",table_name=table_name,layer="BRONZE",file_name=final_file_name)

        # File name validation
        file_name_validation = match_file_name_pattern(file_name_pattern,final_file_name)
        print("file_name_validation =>", file_name_validation)

        # Read summary CSV from Snowflake stage
        df_summary = (session.read.options({"PARSE_HEADER": True,"INFER_SCHEMA": True}).csv(summary_file_path))

        # Trim summary column names
        df_summary = df_summary.to_df(*[c.strip() for c in df_summary.columns])

        # Get summary count for current CSV file
        final_file_name_without_ext = final_file_name.replace(".csv", "")
        summary_count_row = (df_summary.filter(col('"file_name"').contains(lit(final_file_name_without_ext))).select(col('"count"')).first())
        if summary_count_row is None:
            raise ValueError(f"File {final_file_name} not found in summary file")
        summary_count = summary_count_row[0]

        # Read business CSV from Snowflake stage
        df_final_file_name = (session.read.options({"PARSE_HEADER": True,"INFER_SCHEMA": True}).csv(final_file_path))
        table_count = df_final_file_name.count()
        print("summary_count =>", summary_count)
        print("table_count =>", table_count)

        # Validate record count
        if table_count == (int(summary_count) - 1):
            # Load data into Bronze table
            df_final_file_name.write.mode("overwrite").save_as_table(final_table_name)
            data_load_message = "Data load successfully"
            print("data_load_message =>", data_load_message)

            # Get total records inserted
            total_rows = session.table(final_table_name).count()
            print("total_inserted_count =>", total_rows)
            # SUCCESS LOG
            process_log("Success",table_name=table_name,layer="BRONZE",total_inserted_count=total_rows,file_name=final_file_name)
        else:
            data_load_message = ("Summary count is not matched hence process stopped")
            print("data_load_message =>", data_load_message)
            raise ValueError(data_load_message)
    except Exception as e:
        # FAILED LOG
        process_log("Failed",table_name=table_name,layer="BRONZE",error_message=str(e))
        raise